# Llama Metrics

## Joint TKG Quintuples and Triples Calculation

In [ ]:
from metrics import sample_quintuple_compare, sample_triple_compare
from sklearn.metrics import f1_score
from post_processing import get_data, truth_quintuples_and_triples_preprocess, pred_quintuple_preprocess

test_data = get_data("D:\\GeoTKG\\cleandata\\tie\\test.json", truth_quintuples_and_triples_preprocess)
tkg_llama_preds = get_data("llama3-8B-tkg-preds.json", pred_quintuple_preprocess)

In [ ]:
strict_quin_results = []
relaxed_quin_results = []
triple_results = []
for truth, pred in zip(test_data, tkg_llama_preds):
    strict_out, relaxed_out = sample_quintuple_compare(list(truth['quintuples'].values()), list(pred['quintuples'].values()))
    trips_out = sample_triple_compare(truth['triples'], pred['triples'])
    strict_quin_results.extend(strict_out)
    relaxed_quin_results.extend(relaxed_out)
    triple_results.extend(trips_out)

{"Quin relaxed":f1_score([1]*len(relaxed_quin_results), relaxed_quin_results), "Quin strict":f1_score([1]*len(strict_quin_results), strict_quin_results), "Trips Acc":f1_score([1]*len(triple_results), triple_results)}

## Isolated ET NER

In [ ]:
from metrics import sample_ner_compare, get_ner_scores
from post_processing import get_data

test_data = get_data("D:\\GeoTKG\\cleandata\\tie\\test.json")
et_ner_llama_preds = get_data("llama3-8B-et-ner-preds.json")

In [ ]:
strict_ner_et_results = []
relaxed_ner_et_results = []

for pred, truth in zip(et_ner_llama_preds, test_data):
    strict_results, relaxed_results = sample_ner_compare(truth['instances'], pred['pred'])
    strict_ner_et_results.extend(strict_results)
    relaxed_ner_et_results.extend(relaxed_results)
get_ner_scores(strict_ner_et_results, relaxed_ner_et_results)

## Isolated Geo NER

In [ ]:
from post_processing import preprocess_geo_eval, get_data
from metrics import sample_ner_compare, get_ner_scores

test_data = get_data("D:\\GeoTKG\\cleandata\\geo\\eval.json", preprocessor=preprocess_geo_eval)
geo_ner_preds = get_data("D:\\GeoTKG\\experiments\\llama3-8B-geoner-preds.json")

In [ ]:
strict_geo_results = []
relaxed_geo_results = []
for pred, truth in zip(geo_ner_preds, test_data):
    strict, relaxed = sample_ner_compare(truth, pred['pred'], geo_ner=True)
    strict_geo_results.extend(strict)
    relaxed_geo_results.extend(relaxed)
get_ner_scores(strict_geo_results, relaxed_geo_results)

# GeoTKG Metrics

In [1]:
from Pipeline import GeoTKGPipeline
import json
from post_processing import get_data

test_data = get_data("D:\\GeoTKG\\cleandata\\tie\\test.json")

out_preds = []
process_sep = []
model = GeoTKGPipeline()
batch_size = 2
for i in range(0, len(test_data), batch_size):
    samples = test_data[i:i+batch_size]
    for j, sample in enumerate(samples):
        if len(sample['text'])>50:
            process_sep.append(test_data.index(sample))
            samples.pop(j)
    dcts = [inst['value'] for sample in samples for inst in sample['instances'] if inst['type'] != "EVENT" and inst['id'] == 0]
    text = [" ".join([wrd for sent in sample['text'] for wrd in sent]) for sample in samples]
    output = model.pred(text, dcts, return_ner_results=True)
    out_preds.extend(output)
    print(f"Processed {len(out_preds)}/{len(test_data)}")

d:\GeoTKG\venv\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Processed 2/602
Processed 4/602
Processed 6/602
Processed 8/602
Processed 10/602
Processed 12/602
Processed 14/602
Processed 16/602
Processed 18/602
Processed 20/602
Processed 22/602
Processed 24/602
Processed 26/602
Processed 28/602
Processed 30/602
Processed 32/602
Processed 34/602
Processed 36/602
Processed 38/602
Processed 40/602
Processed 42/602
Processed 44/602
Processed 46/602
Processed 48/602
Processed 50/602
Processed 52/602
Processed 54/602
Processed 56/602
Processed 58/602
Processed 60/602
Processed 62/602
Processed 64/602
Processed 66/602
Processed 68/602
Processed 70/602
Processed 72/602
Processed 74/602
Processed 76/602
Processed 78/602
Processed 80/602
Processed 82/602
Processed 84/602
Processed 86/602
Processed 88/602
Processed 90/602
Processed 92/602
Processed 94/602
Processed 96/602
Processed 98/602
Processed 100/602
Processed 102/602
Processed 104/602
Processed 106/602
Processed 108/602
Processed 110/602
Processed 112/602
Processed 114/602
Processed 116/602
Processed

In [2]:
process_sep
for pred_i in process_sep:
    dcts = [inst['value'] for inst in test_data[pred_i]['instances'] if inst['type'] != "EVENT" and inst['id'] == 0]
    text = [" ".join([wrd for sent in test_data[pred_i]['text'] for wrd in sent])]
    output = model.pred(text, dcts, return_ner_results=True)
    out_preds.insert(pred_i, output[0])

In [3]:
from copy import deepcopy
backup = deepcopy(out_preds)

## Joint TKG Quintuples and Triples Calculation

In [ ]:
from metrics import sample_quintuple_compare, sample_triple_compare
from sklearn.metrics import f1_score
from post_processing import get_data, truth_quintuples_and_triples_formating

test_data = get_data("D:\\GeoTKG\\cleandata\\tie\\test.json", truth_quintuples_and_triples_formating)

strict_results = []
relaxed_results = []
triples_results = []
for truth, pred in zip(test_data, out_preds):
    strict_out, relaxed_out = sample_quintuple_compare(list(truth['quintuples'].values()), pred['quintuples'])
    relaxed_results.extend(relaxed_out)
    strict_results.extend(strict_out)
    trips_out = sample_triple_compare(truth['triples'], pred['triples'])
    triples_results.extend(trips_out)

{"Quin relaxed":f1_score([1]*len(relaxed_results), relaxed_results), "Quin strict":f1_score([1]*len(strict_results), strict_results), "triples":f1_score([1]*len(triples_results), triples_results)}

## Isolated ET NER

In [4]:
from post_processing import get_data
from metrics import sample_ner_compare, get_ner_scores

test_data = get_data("D:\\GeoTKG\\cleandata\\tie\\test.json")

In [8]:
strict_ner_et_results = []
relaxed_ner_et_results = []
for pred, truth in zip(out_preds, test_data):
    strict_results, relaxed_results = sample_ner_compare(truth['instances'], pred['events']+pred['times'])
    strict_ner_et_results.extend(strict_results)
    relaxed_ner_et_results.extend(relaxed_results)
get_ner_scores(strict_ner_et_results, relaxed_ner_et_results)

{'strict_text': 0.8772106639790752,
 'relaxed_text': 0.886853499547985,
 'type': 0.8839328308367522}

## Isolated Normalisation
Done in training file

## Isolated ET Linking and EE Temporal Relations
Done in training file